# Efficacité Énergétique des Bâtiments — Prédiction DPE + Régression

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 6 000 bâtiments (France)

### Plan
1. Exploration — distribution DPE
2. Variables influentes sur la consommation
3. ColumnTransformer (num + OHE + ordinal)
4. Comparaison modèles
5. Évaluation
6. Importance des variables
7. Simulation de rénovation

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_generation import load_or_generate
from src.energy_model import (build_pipeline, compare_models, evaluate_model,
                               plot_dpe_distribution, plot_feature_importance,
                               NUM_FEATURES, OHE_FEATURES, ORD_FEATURES, TARGET)
print('Imports OK')

## 1. Exploration — Labels DPE

In [ ]:
df = load_or_generate('../data_sample/building_energy_simulated.csv')
print(f'Bâtiments : {len(df):,} | Conso moy : {df.conso_kwh_m2.mean():.1f} kWh/m²/an')
print(df['label_dpe'].value_counts().sort_index())
df.head()

In [ ]:
plot_dpe_distribution(df)

## 2. Analyse par type d'isolation et de vitrage

In [ ]:
print('Consommation moy par isolation :')
print(df.groupby('isolation')['conso_kwh_m2'].mean().round(1))
print('\nConsommation moy par vitrage :')
print(df.groupby('vitrage')['conso_kwh_m2'].mean().round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.boxplot(column='conso_kwh_m2', by='isolation', ax=axes[0])
df.boxplot(column='conso_kwh_m2', by='type_batiment', ax=axes[1])
df.boxplot(column='conso_kwh_m2', by='chauffage', ax=axes[2])
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.suptitle('Consommation par catégorie', fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Modèle — Comparaison CV

In [ ]:
X = df[NUM_FEATURES + OHE_FEATURES + ORD_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('Cross-validation R² :')
cv_res = compare_models(X_train, y_train)

## 4. Entraînement & Évaluation

In [ ]:
pipe = build_pipeline('xgb')
pipe.fit(X_train, y_train)
metrics = evaluate_model(pipe, X_test, y_test)

## 5. Importance des variables

In [ ]:
plot_feature_importance(pipe)

## 6. Simulation — Impact d'une rénovation

In [ ]:
# Simuler un bâtiment avant/après rénovation
exemple = X_test.iloc[[0]].copy()
conso_avant = pipe.predict(exemple)[0]

# Après rénovation : meilleure isolation + vitrage triple
exemple_renove = exemple.copy()
exemple_renove['isolation'] = 'tres_fort'
exemple_renove['vitrage'] = 'triple'
conso_apres = pipe.predict(exemple_renove)[0]

economie = conso_avant - conso_apres
print(f'Conso avant rénovation : {conso_avant:.1f} kWh/m²')
print(f'Conso après rénovation  : {conso_apres:.1f} kWh/m²')
print(f'Économie estimée        : {economie:.1f} kWh/m² (-{economie/conso_avant:.1%})')

## Synthèse

| Métrique | Valeur |
|----------|--------|
| MAE | ~10 kWh/m² |
| RMSE | ~15 kWh/m² |
| R² | ~0.88 |

> **Top variables :** isolation, age_construction, vitrage, type_bâtiment  

*Données simulées (seed=42) — Emmanuel TSAGUE*